# Readout ping — fire a tone at 2.76 GHz and plot the `robs` trace

Fire **one readout-drive pulse** on **qubit 1, readout channel (1)** at **2.76 GHz** and read back
the trace the SoC captured while it played. The **hardware** sibling of
[`remote_pulse.ipynb`](remote_pulse.ipynb) / [`iq_scatter.ipynb`](iq_scatter.ipynb): it drives a
real ZCU216 with the `xm650-loopback` build over `RemoteDriver`, so it is **not executed in CI**.

**What `robs` is.** On *any* core's readout-drive pulse fire, the SoC streams the per-lane sum of
the **mapped ADC inputs** into one shared readout-observation buffer (`robs`) for the whole time the
pulse is valid — a raw ADC-rate trace of what the converters saw during the readout window, addressed
by a fire-incremented pointer and read straight back over AXI (`riscq.run.read_robs`). With the
loopback cable in place, that trace *is* the readout tone coming back off the DAC.

So the recipe is: play a readout pulse on qubit 1, let its `[startTime, startTime+dur)` window drive
the `robs` capture, then plot what came back. No demod is needed — the readout-drive pulse's own
valid window is what triggers the trace.

In [ ]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

from riscq import run as rq
from riscq.driver.remote import RemoteDriver, upload_bundle
from riscq.lang import Array, ParamTable, compile_kernel, kernel
from riscq.map import ADC_BATCH, LEAD, SocMap, SocParams
from riscq.pulses import Pulse, envelopes, units

BOARD = "192.168.1.122"                   # the ZCU216's LAN address (or the full PYRO: uri)

drv = RemoteDriver(BOARD, 9091)
print("server:", drv.board.info())

## The bundle

This uses the 2-core `xm650-loopback` build
([`software/configs/xm650-loopback.json`](../software/configs/xm650-loopback.json)): each core's
gate + readout drive sum onto its **own DAC** (qubit 0 → DAC 0, qubit 1 → DAC 1) and its demod
listens on its **own ADC** (qubit 0 → ADC 14, qubit 1 → ADC 15). A cable from **qubit 1's DAC (1)
to its ADC (15)** closes the loop so the readout tone comes back. The `robs` trace sums the mapped
ADCs (14 + 15), so with only qubit 1 driving, it shows exactly qubit 1's returned tone.

(To target qubit 0 instead — the DAC 0 → ADC 14 loop [`remote_pulse.ipynb`](remote_pulse.ipynb)
documents — set `QUBIT = 0` below.)

The upload is needed **once per build**; `load` is needed once per server start (ref clocks →
overlay → MTS → Nyquist zones). Skip both if the server already reports the bundle loaded above.

In [ ]:
print("bundles on the board:", drv.board.bundles())

# first time only — push the build up and load it (~100 MB, a minute on GbE):
# upload_bundle(drv, "xm650-loopback",
#               xsa="../build/xm650-loopback/top.xsa",                    # write_hw_platform export
#               params_json="../software/configs/xm650-loopback.json")   # the SAME JSON the build used
# info = drv.board.load("xm650-loopback")                                # full RF bring-up; returns info()
# print(info)
# assert info["mts_result"] == 0, "multi-tile sync missed its target latencies"

m = SocMap(SocParams.from_json(drv.board.get_params()))   # always matches the loaded bitstream
dac_fs = units.sample_rate(m.params)
adc_fs = ADC_BATCH * m.params.dsp_freq_hz                  # ADC sample rate (4 samples/batch)
print(f"connected to '{m.params.name}': {m.params.qubit_num} cores, "
      f"{dac_fs / 1e9:.0f} GS/s DACs, {adc_fs / 1e9:.0f} GS/s ADCs")

## The kernel

One readout-drive pulse on channel 1 (this core's readout DAC), fired `LEAD` batches into the
future. `init_pulse_params` loads the readout slot, `set_freq` programs the 2.76 GHz carrier, and
`play` emits the tone — its valid window is what streams the ADC trace into `robs`. `wait_until`
holds the core past the window so the capture finishes before the kernel reports DONE.

In [ ]:
QUBIT = 1                                 # qubit 1 = core index 1 (its DAC 1 loops to ADC 15)
F_READOUT = 2.76e9                        # readout carrier (Hz)
WIN = 40                                  # readout window in batches = robs rows captured


@kernel
def readout_ping(ro: ParamTable, ts: Array):
    """Fire ONE readout-drive pulse; the SoC captures the looped-back ADC trace into `robs`
    for the whole time the pulse is valid."""
    init_pulse_params(ro.pulses)                     # noqa: F821  load the readout-drive slot
    set_freq(ro, ro.freq)                            # noqa: F821  program the 2.76 GHz carrier
    t = now() + LEAD                                 # noqa: F821  schedule LEAD batches ahead
    ts[0] = t                                         # record the fire time for the host
    play(ro, ro["meas"], t)                          # noqa: F821  readout tone -> DAC; drives robs
    wait_until(t + ro["meas"].dur + 32)              # noqa: F821  hold past the capture window


# a flat square readout tone; channel 1 stores 16 samples/line at this build's interp = 1
ro = ParamTable(1, F_READOUT, {"meas": Pulse(envelopes.square(WIN * 16), freq_hz=F_READOUT, amp=0.8)})
prog = compile_kernel(readout_ping, m, tables=dict(ro=ro), ts=Array(1))

dur = ro.pulses["meas"].dur_batches(m, 1)             # readout window in batches = robs rows
print(f"compiled: readout pulse {dur} batches ({units.ns(dur, m.params):.0f} ns) at "
      f"{F_READOUT / 1e9:.2f} GHz on qubit {QUBIT}, channel 1 (code {units.freq_to_code(F_READOUT, m.params)})")

## Run it and read `robs`

`rq.run` = `setup` (load the image + envelope + pulse table, park the other core) + one `rerun`
(release reset → the kernel fires the pulse → poll DONE → re-assert reset). The pulse's valid window
streams the trace into `robs`; `rq.read_robs` then fetches the whole shared buffer over AXI — it
holds the last window's capture, so we slice the first `dur` rows.

In [ ]:
out = rq.run(drv, m, {QUBIT: prog}, results=["ts"], timeout=5_000)   # setup + one fire; parks the other core
print("fired readout pulse at batch", int(out[QUBIT]["ts"][0]))

rob = rq.read_robs(drv, m).reshape(-1, ADC_BATCH)    # (rob_depth, 4) int32 lanes — one row per batch
trace = rob[:dur].reshape(-1)                         # the captured window, in ADC-sample order
t_ns = np.arange(trace.size) / adc_fs * 1e9

print(f"robs: {dur} batches x {ADC_BATCH} lanes = {trace.size} ADC samples "
      f"({units.ns(dur, m.params):.0f} ns at {adc_fs / 1e9:.0f} GS/s)")
print(f"trace peak |code| = {int(np.abs(trace).max())}")

## Plot the trace

The left panel is the whole readout window; the right zooms into the first samples. The ADC runs at
2 GS/s (Nyquist 1 GHz), so the 2.76 GHz drive **folds to ~0.76 GHz** in the raw trace — these are the
exact samples the demod integrates against its (4×) carrier code. A flat-noise trace here means the
loopback cable for qubit 1 (DAC 1 → ADC 15) isn't in place.

In [ ]:
alias = abs(((F_READOUT + adc_fs / 2) % adc_fs) - adc_fs / 2)   # tone folded into the ADC's 1st Nyquist

fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(11, 4))

ax_full.plot(t_ns, trace, lw=0.7, color="#1f77b4")
ax_full.set_xlabel("time (ns)"); ax_full.set_ylabel("ADC code (mapped-ADC sum)")
ax_full.set_title(f"robs — {dur}-batch readout window @ {F_READOUT / 1e9:.2f} GHz")

nz = min(48, trace.size)
ax_zoom.plot(t_ns[:nz], trace[:nz], "o-", ms=3, lw=0.8, color="#d62728")
ax_zoom.set_xlabel("time (ns)"); ax_zoom.set_ylabel("ADC code")
ax_zoom.set_title(f"first {nz} samples (folds to ~{alias / 1e9:.2f} GHz at {adc_fs / 1e9:.0f} GS/s)")

for ax in (ax_full, ax_zoom):
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Done

`robs` is a diagnostic trace, not the measurement path — for calibrated single-shot `(I, Q)`
integrals see [`iq_scatter.ipynb`](iq_scatter.ipynb), and for a frequency sweep
[`vna.ipynb`](vna.ipynb).

In [ ]:
drv.close()
print("done")